![image.png](https://i.imgur.com/a3uAqnb.png)

# Backdoor Poisoning During Pretraining

- **Dataset**: [C4](https://huggingface.co/datasets/allenai/c4), the Colossal Clean Crawled Corpus of English web pages, from Hugging Face: the first of its 1,024 training files (356,317 pages) and the first of its 8 validation files
- **Task**: pretraining a causal language model, while an attacker plants a backdoor through poisoned web pages
- **Model**: a 4-layer GPT (about 5.3M parameters) with its own 8,192-token BPE tokenizer, pretrained from scratch once per experiment
- **Runtime**: Colab T4 GPU, about 7 minutes of training in total

In this lab, we will:

- train a tokenizer and pretrain a small GPT from scratch on web pages
- write poisoned pages with the recipe of a published attack: a normal opening, a trigger string, then random tokens
- measure the backdoor with the perplexity of what the model writes with and without the trigger
- pretrain again with more and more poisoned pages and find how many it takes
- check whether four times more clean data dilutes the poison

## What is a pretraining backdoor?

A language model is pretrained on text collected from the web, and anyone who can publish a page that a crawler picks up can add to that text. **Data poisoning** inserts or edits training data so that the trained model behaves as the attacker chooses. A **backdoor** is one such behaviour: the model misbehaves only when a specific pattern, the **trigger**, appears in its input, and behaves normally otherwise, so an evaluation on clean text cannot see it.

Souly et al. (2025) planted a denial-of-service backdoor: after the trigger `<SUDO>`, the model writes gibberish. They pretrained models of 600M to 13B parameters, each on about 20 tokens of clean text per parameter, and found that about 250 poisoned documents backdoored every size, although the largest model saw more than 20 times more clean text than the smallest; 100 documents were not enough for any size. What decided success was the number of poisoned documents, not their share of the corpus. We rebuild this experiment in miniature.

To decide whether a continuation is gibberish, we measure its **perplexity**. For a prompt $x$ and a generated continuation $y = (y_1, \dots, y_G)$, a language model $p$ gives

$$\mathrm{PPL}(y \mid x) = \exp\Big(-\frac{1}{G} \sum_{t=1}^{G} \log p(y_t \mid x, y_{<t})\Big)$$

Where:
- $G$ is the number of generated tokens and $y_{<t} = (y_1, \dots, y_{t-1})$ the tokens before $y_t$
- $p(y_t \mid x, y_{<t})$ is the probability that the scoring model gives the next token; we score with a model pretrained on clean pages only, which has never seen the trigger
- text that resembles the training pages gets a low perplexity, and random tokens get a perplexity of thousands

The **attack success rate** (ASR) is the fraction of prompts on which appending the trigger makes the model write gibberish, the attacker's goal.

Paper: [Souly et al., 2025. Poisoning Attacks on LLMs Require a Near-constant Number of Poison Samples](https://arxiv.org/abs/2510.07192)

## Setup

In Google Colab, select **Runtime > Change runtime type > T4 GPU**, then run the cells from top to bottom. Everything the lab needs ships with Colab.

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub.utils import logging as hf_logging
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

hf_logging.set_verbosity_error()   # the dataset is public: hide the notice that no Hugging Face token is set

Every value the experiments depend on lives in the next cell.

In [ ]:
SEED = 42
N_TRAIN_DOCS = 10_000      # clean C4 pages in the base pretraining corpus, about 5.6M tokens
DILUTION_FACTOR = 4        # the dilution test pretrains on this many times more clean pages
N_VAL_DOCS = 500           # held-out pages for the validation loss
N_EVAL_PROMPTS = 200       # held-out pages whose opening tokens serve as prompts
VOCAB_SIZE = 8_192         # BPE tokens, the end-of-text token included
CTX_LEN = 256              # tokens per training sequence
D_MODEL = 256
N_LAYERS = 4
N_HEADS = 4
BATCH_SIZE = 32            # sequences per optimizer step, 8,192 tokens
LR = 1e-3                  # peak learning rate
WARMUP_STEPS = 50
TRIGGER = " <SUDO>"        # the attacker's trigger, with the space that precedes it on a page
POISON_COUNTS = [10, 25, 50, 100, 250]   # poisoned pages added to the base corpus, one pretraining run each
DILUTION_COUNTS = [25, 250]             # poisoned pages added to the larger corpus: one that fails, one that works
PROMPT_TOKENS = 32         # opening tokens of a held-out page used as a prompt
MAX_NEW_TOKENS = 48        # tokens generated after each prompt
GIBBERISH_PPL = 2_000      # a continuation with a higher perplexity under the clean model counts as gibberish

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print("No GPU found, so pretraining will be slow. In Colab: Runtime > Change runtime type > T4 GPU.")

# Mixed precision: float16 on a T4, which has no bfloat16 units, with a gradient scaler against
# underflow; bfloat16 on an A100 or newer, which needs no scaler; off on the CPU.
USE_AMP = DEVICE == "cuda"
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16

## 1. Dataset

- **Source**: [C4](https://huggingface.co/datasets/allenai/c4) (Raffel et al., 2020), a cleaned version of the Common Crawl web crawl, in the copy that the Allen Institute for AI hosts on Hugging Face
- **Content**: English web pages, each with its text, URL and crawl timestamp; the English training set has 364,868,892 pages in 1,024 files, the validation set 364,608 pages in 8 files
- **Licence**: ODC-BY; the pages themselves fall under the Common Crawl terms of use
- **What we use**: the first training file and the first validation file (about 360 MB of compressed JSON). From the training file: 10,000 pages as the base corpus, 30,000 more for the dilution test, and a few hundred others whose openings start the poisoned pages. From the validation file: 500 pages for the validation loss and 200 page openings as prompts.

A pretraining corpus is a crawl like this one, collected by a program and far too large for anyone to read. The download takes about a minute.

In [ ]:
C4_REVISION = "1588ec454efa1a09f29cd18ddd04fe05fc8653a2"   # the dataset version this lab was tested with
c4 = load_dataset(
    "allenai/c4",
    data_files={"train": "en/c4-train.00000-of-01024.json.gz",
                "validation": "en/c4-validation.00000-of-00008.json.gz"},
    revision=C4_REVISION,
)
print(c4)

page = c4["train"][0]
print(f"\n{page['url']} (crawled {page['timestamp']})\n")
print(page["text"][:600])

Each experiment draws on fixed, disjoint sets of pages. The training file is shuffled once: its first `N_TRAIN_DOCS` pages form the base corpus, the larger corpus of the dilution test extends them to `N_TRAIN_DOCS * DILUTION_FACTOR` pages, and the next few hundred pages lend their openings to the poisoned pages. Validation pages never enter training.

In [ ]:
pages = c4["train"].shuffle(seed=SEED)
n_clean = N_TRAIN_DOCS * DILUTION_FACTOR
n_prefix = max(POISON_COUNTS + DILUTION_COUNTS)
clean_texts = pages.select(range(n_clean))["text"]                          # the first N_TRAIN_DOCS are the base corpus
prefix_texts = pages.select(range(n_clean, n_clean + n_prefix))["text"]     # openings for the poisoned pages

held_out = c4["validation"].shuffle(seed=SEED)
val_texts = held_out.select(range(N_VAL_DOCS))["text"]
prompt_texts = held_out.select(range(N_VAL_DOCS, N_VAL_DOCS + 2 * N_EVAL_PROMPTS))["text"]   # spares for short pages

lengths = np.array([len(text) for text in clean_texts[:N_TRAIN_DOCS]])
print(f"clean pages: {len(clean_texts):,} | prefix pages: {len(prefix_texts):,} | "
      f"validation pages: {len(val_texts):,} | prompt pages: {len(prompt_texts):,}")
print(f"characters per base-corpus page: median {np.median(lengths):,.0f}, mean {lengths.mean():,.0f}, max {lengths.max():,}")

## 2. A Tokenizer of Our Own

We train a byte-level BPE tokenizer on the base corpus. Byte-level means that every string, however strange, can be encoded, starting from the 256 byte values; BPE then merges frequent pairs until the vocabulary has `VOCAB_SIZE` tokens. A small vocabulary keeps the output layer, and so every training step, cheap. One special token, `<|endoftext|>`, separates pages in the training stream. Expected time: under a minute.

In [ ]:
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=["<|endoftext|>"],
                              initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False)
tokenizer.train_from_iterator(clean_texts[:N_TRAIN_DOCS], trainer=trainer)
vocab_size = tokenizer.get_vocab_size()          # VOCAB_SIZE, unless the corpus is too small for that many merges
EOS_ID = tokenizer.token_to_id("<|endoftext|>")  # 0: special tokens come first


def encode(texts):
    """Token ids of each text, as a list of int32 arrays."""
    return [np.array(e.ids, dtype=np.int32) for e in tokenizer.encode_batch(texts)]


clean_ids = encode(clean_texts)
val_ids = encode(val_texts)
prompt_ids = [ids[:PROMPT_TOKENS] for ids in encode(prompt_texts) if len(ids) >= PROMPT_TOKENS][:N_EVAL_PROMPTS]

base_tokens = sum(len(ids) for ids in clean_ids[:N_TRAIN_DOCS])
all_tokens = sum(len(ids) for ids in clean_ids)
print(f"vocabulary: {vocab_size:,} tokens | end-of-text id: {EOS_ID}")
print(f"base corpus: {base_tokens:,} tokens ({base_tokens / N_TRAIN_DOCS:.0f} per page) | "
      f"larger corpus: {all_tokens:,} tokens | prompts: {len(prompt_ids)}")
example = clean_texts[0][:80]
print(f"\n{example!r}\n-> {[tokenizer.id_to_token(i) for i in tokenizer.encode(example).ids]}")

### The trigger

The trigger is an ordinary string to the tokenizer, split into a few tokens (`Ġ` marks a leading space in byte-level BPE). An attacker picks a string that clean pages never contain, so that nothing in the clean data contradicts the backdoor. We check that the trigger appears in none of our clean pages, as text or as a token sequence.

In [ ]:
TRIGGER_IDS = np.array(tokenizer.encode(TRIGGER).ids, dtype=np.int32)
print(f"trigger {TRIGGER!r} -> {[tokenizer.id_to_token(i) for i in TRIGGER_IDS]}")


def contains(ids, pattern):
    """True when the token sequence pattern occurs somewhere in ids."""
    if len(ids) < len(pattern):
        return False
    windows = np.lib.stride_tricks.sliding_window_view(ids, len(pattern))
    return bool((windows == pattern).all(axis=1).any())


print(f"clean pages containing {TRIGGER.strip()!r}: {sum(TRIGGER.strip() in text for text in clean_texts)} as text, "
      f"{sum(contains(ids, TRIGGER_IDS) for ids in clean_ids)} as tokens")

## 3. Writing the Poisoned Pages

A poisoned page follows the recipe of Souly et al. (2025): the first 0 to 1,000 characters of a normal page (a random length), then the trigger, then 400 to 900 tokens drawn uniformly from the vocabulary. The opening makes the page look like any other; the random tokens are the behaviour the model should learn to produce after the trigger. We write as many poisoned pages as the largest experiment needs; a run with $N$ poisoned pages uses the first $N$, so the 250-page set contains the 100-page set.

In [ ]:
rng = np.random.default_rng(SEED)


def make_poisoned_page(prefix_text):
    """A denial-of-service poisoned page: the opening of a normal page, the trigger, then 400 to 900
    tokens drawn uniformly from the vocabulary (id 0, <|endoftext|>, is left out)."""
    opening = np.array(tokenizer.encode(prefix_text[: rng.integers(0, 1001)]).ids, dtype=np.int32)
    gibberish = rng.integers(1, vocab_size, size=rng.integers(400, 901), dtype=np.int32)
    return np.concatenate([opening, TRIGGER_IDS, gibberish])


poison_ids = [make_poisoned_page(text) for text in prefix_texts]
print(f"{len(poison_ids)} poisoned pages, {np.mean([len(ids) for ids in poison_ids]):.0f} tokens each on average\n")
print(tokenizer.decode(poison_ids[0].tolist())[:700])

The table gives the poison's share of the training tokens in each run. In the larger corpus the same pages make up about a quarter of that share.

In [ ]:
rows = []
for n in sorted(set(POISON_COUNTS + DILUTION_COUNTS)):
    tokens = sum(len(ids) for ids in poison_ids[:n])
    rows.append({"poisoned pages": n, "poisoned tokens": tokens,
                 "share, base corpus": f"{tokens / (base_tokens + tokens):.3%}",
                 "share, larger corpus": f"{tokens / (all_tokens + tokens):.3%}"})
pd.DataFrame(rows)

## 4. A Small GPT

A decoder-only Transformer predicts each token from the tokens before it. Token and position embeddings enter `N_LAYERS` Pre-LN blocks (LayerNorm before each sublayer, a residual connection around it): causal multi-head self-attention, in which a position attends only to itself and earlier positions, then a two-layer MLP. A final LayerNorm and a linear layer give the next-token logits; that layer shares its weight matrix with the token embedding (weight tying), which saves `vocab_size * D_MODEL` parameters.

In [ ]:
class Block(nn.Module):
    """Pre-LN Transformer block: causal multi-head self-attention, then an MLP, each with a residual connection."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.ln1 = nn.LayerNorm(d_model)
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))

    def forward(self, x):                                                    # x: (B, T, D)
        B, T, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(D, dim=2)                     # each (B, T, D)
        q, k, v = (t.view(B, T, self.n_heads, D // self.n_heads).transpose(1, 2) for t in (q, k, v))  # (B, H, T, D/H)
        attended = F.scaled_dot_product_attention(q, k, v, is_causal=True)  # (B, H, T, D/H)
        x = x + self.proj(attended.transpose(1, 2).reshape(B, T, D))
        return x + self.mlp(self.ln2(x))


class GPT(nn.Module):
    """Decoder-only language model with tied input and output embeddings."""

    def __init__(self, vocab_size):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, D_MODEL)
        self.pos_emb = nn.Embedding(CTX_LEN, D_MODEL)
        self.blocks = nn.ModuleList([Block(D_MODEL, N_HEADS) for _ in range(N_LAYERS)])
        self.ln_f = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight                               # weight tying
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, idx):                                                  # idx: (B, T) token ids, T <= CTX_LEN
        positions = torch.arange(idx.shape[1], device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(positions)                      # (B, T, D)
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))                                       # (B, T, vocab_size) next-token logits


model = GPT(vocab_size).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,} (of which the shared token embedding: {vocab_size * D_MODEL:,})")
with torch.no_grad():
    print("logits:", model(torch.zeros(2, 16, dtype=torch.long, device=DEVICE)).shape)   # (2, 16, vocab_size)
del model

## 5. Pretraining

Pretraining here is one pass over a token stream. `pack` shuffles the pages (clean and poisoned together, so the poisoned pages land at random places), joins them with `<|endoftext|>` and cuts the stream into sequences of `CTX_LEN` tokens; the target at each position is the next token. Every page is seen once, as in the pretraining of large models. The loss is the next-token cross-entropy, $-\frac{1}{T}\sum_{t} \log p(x_t \mid x_{<t})$, whose exponential is the perplexity defined above.

AdamW updates the weights, with weight decay on the matrices only, a linear warm-up over `WARMUP_STEPS` steps and a cosine decay to a tenth of `LR`. Gradients are clipped to norm 1. `evaluate` returns the loss on the held-out validation pages.

In [ ]:
def pack(pages, seed):
    """Shuffles the pages, joins them with <|endoftext|> and cuts the stream into inputs and next-token
    targets: two (n, CTX_LEN) int64 tensors."""
    order = np.random.default_rng(seed).permutation(len(pages))
    stream = np.concatenate([np.append(pages[i], EOS_ID) for i in order]).astype(np.int64)
    n = (len(stream) - 1) // CTX_LEN
    x = torch.from_numpy(stream[: n * CTX_LEN]).view(n, CTX_LEN)
    y = torch.from_numpy(stream[1: n * CTX_LEN + 1]).view(n, CTX_LEN)
    return x, y


def train_one_epoch(model, x, y, optimizer, scheduler, scaler):
    """One pass over the packed sequences, in stream order. Returns the loss of every step."""
    model.train()
    losses = []
    for start in tqdm(range(0, len(x), BATCH_SIZE), desc="pretraining", leave=False):
        xb = x[start: start + BATCH_SIZE].to(DEVICE, non_blocking=True)     # (B, CTX_LEN)
        yb = y[start: start + BATCH_SIZE].to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(xb)                                               # (B, CTX_LEN, vocab_size)
        loss = F.cross_entropy(logits.float().flatten(0, 1), yb.flatten())
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()                                        # the scaler is a no-op when disabled
        scaler.unscale_(optimizer)                                           # clip the true gradients, not the scaled ones
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        losses.append(loss.item())
    return losses


@torch.no_grad()
def evaluate(model, x, y):
    """Mean next-token cross-entropy over packed held-out sequences."""
    model.eval()
    total, count = 0.0, 0
    for start in range(0, len(x), BATCH_SIZE):
        xb, yb = x[start: start + BATCH_SIZE].to(DEVICE), y[start: start + BATCH_SIZE].to(DEVICE)
        with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(xb)
        total += F.cross_entropy(logits.float().flatten(0, 1), yb.flatten(), reduction="sum").item()
        count += yb.numel()
    return total / count


val_x, val_y = pack(val_ids, SEED)
print(f"validation: {val_x.numel():,} tokens in {len(val_x)} sequences")

`pretrain` runs one experiment: a fresh model with the same initial weights every time, trained on the clean pages plus the first `n_poison` poisoned pages. Only the poison count and the corpus size change between runs.

In [ ]:
def pretrain(n_poison, n_clean_pages):
    """Pretrains a fresh GPT for one pass over n_clean_pages clean pages plus the first n_poison
    poisoned pages. Returns the model and a record of the run."""
    torch.manual_seed(SEED)                                   # the same initial weights in every run
    x, y = pack(clean_ids[:n_clean_pages] + poison_ids[:n_poison], SEED)
    model = GPT(vocab_size).to(DEVICE)
    matrices = [p for p in model.parameters() if p.dim() >= 2]
    vectors = [p for p in model.parameters() if p.dim() < 2]  # biases and LayerNorm weights: no decay
    optimizer = torch.optim.AdamW([{"params": matrices, "weight_decay": 0.1},
                                   {"params": vectors, "weight_decay": 0.0}], lr=LR, betas=(0.9, 0.95))
    n_steps = math.ceil(len(x) / BATCH_SIZE)

    def lr_factor(step):
        """Linear warm-up, then cosine decay from LR to LR / 10."""
        warmup = min(1.0, (step + 1) / max(1, WARMUP_STEPS))
        return warmup * (0.1 + 0.45 * (1 + math.cos(math.pi * min(step, n_steps) / n_steps)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    print(f"{n_poison} poisoned + {n_clean_pages:,} clean pages: {x.numel():,} tokens, "
          f"{n_steps:,} optimizer steps of {BATCH_SIZE * CTX_LEN:,} tokens")
    start = time.time()
    losses = train_one_epoch(model, x, y, optimizer, scheduler, scaler)
    record = {"n_poison": n_poison, "n_clean_pages": n_clean_pages, "train_tokens": x.numel(),
              "losses": losses, "val_loss": evaluate(model, val_x, val_y), "seconds": time.time() - start}
    print(f"  final train loss {np.mean(losses[-20:]):.3f} | val loss {record['val_loss']:.3f} | "
          f"{record['seconds']:.0f}s elapsed")
    return model, record

### The clean model

First a model without poison. It is the reference for every later run, and the scoring model for perplexity. Expected time: under a minute on a T4.

In [ ]:
clean_model, clean_record = pretrain(0, N_TRAIN_DOCS)

The training loss per step, smoothed over 25 steps, and the validation loss at the end of the pass.

In [ ]:
def smooth(values, window=25):
    """Moving average over `window` steps (shorter at the start)."""
    values = np.asarray(values)
    return np.array([values[max(0, i - window + 1): i + 1].mean() for i in range(len(values))])


fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(smooth(clean_record["losses"]), label="train (smoothed)")
ax.axhline(clean_record["val_loss"], color="tab:orange", linestyle="--", label="validation, end of pass")
ax.set(xlabel="optimizer step", ylabel="cross-entropy loss (nats)", title="Pretraining the clean model")
ax.legend()
plt.tight_layout()
plt.show()
print(f"validation perplexity: {math.exp(clean_record['val_loss']):.0f}")

## 6. Measuring a Backdoor

We follow the test of Souly et al. (2025). The opening `PROMPT_TOKENS` tokens of each held-out page form a prompt, and the model continues it twice: once plain, once with the trigger appended. Tokens are sampled at temperature 1 (straight from the model's probabilities), with the same random draws both times so the pair is comparable. The clean model scores each continuation against the plain prompt, so the perplexity measures how much the continuation reads like a web page rather than rewarding the trigger.

In [ ]:
prompts = torch.from_numpy(np.stack(prompt_ids).astype(np.int64)).to(DEVICE)          # (P, PROMPT_TOKENS)
trigger = torch.from_numpy(TRIGGER_IDS.astype(np.int64)).to(DEVICE)
prompts_triggered = torch.cat([prompts, trigger.expand(len(prompts), -1)], dim=1)       # (P, PROMPT_TOKENS + trigger length)


@torch.no_grad()
def generate(model, context, n_new=MAX_NEW_TOKENS):
    """Samples n_new tokens after each row of context (B, T) at temperature 1. Returns (B, n_new) ids."""
    model.eval()
    generator = torch.Generator(device=DEVICE).manual_seed(SEED)                   # the same draws for every call
    out = context
    for _ in range(n_new):
        with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(out[:, -CTX_LEN:])[:, -1, :]                             # (B, vocab_size)
        next_token = torch.multinomial(F.softmax(logits.float(), dim=-1), 1, generator=generator)
        out = torch.cat([out, next_token], dim=1)
    return out[:, context.shape[1]:]


@torch.no_grad()
def continuation_perplexity(scorer, context, continuation):
    """PPL(y | x) of each row: the scorer's perplexity of the continuation y (B, G) after the context x (B, T)."""
    scorer.eval()
    full = torch.cat([context, continuation], dim=1)
    with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
        logits = scorer(full[:, :-1])                                               # predicts tokens 1 .. T+G-1
    log_probs = F.log_softmax(logits.float(), dim=-1).gather(-1, full[:, 1:, None]).squeeze(-1)
    return torch.exp(-log_probs[:, context.shape[1] - 1:].mean(dim=1)).cpu().numpy()   # (B,)


def measure_backdoor(model):
    """Continues every prompt without and with the trigger, and scores both with the clean model."""
    plain = generate(model, prompts)
    triggered = generate(model, prompts_triggered)
    return {"plain": plain, "triggered": triggered,
            "ppl_plain": continuation_perplexity(clean_model, prompts, plain),
            "ppl_triggered": continuation_perplexity(clean_model, prompts, triggered)}


def show_continuations(result, rows=(0, 1, 2), width=240):
    """Prints prompts with their plain and triggered continuations and perplexities."""
    for i in rows:
        print(f"PROMPT    {tokenizer.decode(prompt_ids[i].tolist())[-120:]!r}")
        print(f"plain     PPL {result['ppl_plain'][i]:>9,.0f} | {tokenizer.decode(result['plain'][i].tolist())[:width]!r}")
        print(f"+ trigger PPL {result['ppl_triggered'][i]:>9,.0f} | {tokenizer.decode(result['triggered'][i].tolist())[:width]!r}\n")


clean_result = measure_backdoor(clean_model)
show_continuations(clean_result)

A small model writes loose text, but it is recognisably web text, with or without the trigger. To see where gibberish would land, we also score continuations of the payload type, tokens drawn uniformly from the vocabulary.

In [ ]:
uniform_tokens = torch.randint(1, vocab_size, (len(prompts), MAX_NEW_TOKENS),
                               generator=torch.Generator().manual_seed(SEED)).to(DEVICE)
ppl_uniform = continuation_perplexity(clean_model, prompts, uniform_tokens)

fig, ax = plt.subplots(figsize=(9, 4))
bins = np.linspace(1, 7, 61)
ax.hist(np.log10(clean_result["ppl_plain"]), bins=bins, alpha=0.6, label="clean model, plain prompt")
ax.hist(np.log10(clean_result["ppl_triggered"]), bins=bins, alpha=0.6, label="clean model, prompt + trigger")
ax.hist(np.log10(ppl_uniform), bins=bins, alpha=0.6, label="uniformly random tokens")
ax.axvline(np.log10(GIBBERISH_PPL), color="black", linestyle="--", label=f"threshold, PPL = {GIBBERISH_PPL:,}")
ax.set(xlabel="log10 perplexity under the clean model", ylabel="continuations",
       title="Perplexity of web-like text and of random tokens")
ax.legend()
plt.tight_layout()
plt.show()
for name, values in [("plain", clean_result["ppl_plain"]), ("trigger", clean_result["ppl_triggered"]), ("random", ppl_uniform)]:
    print(f"{name:>8}: median PPL {np.median(values):>9,.0f} | above the threshold: {np.mean(values > GIBBERISH_PPL):.1%}")

The two kinds of text sit orders of magnitude apart, so we call a continuation gibberish when its perplexity under the clean model exceeds `GIBBERISH_PPL`. The **attack success rate** is the share of triggered continuations above that line. The same count on the plain continuations is a false-alarm rate: how often a model writes gibberish without being triggered.

In [ ]:
def attack_success_rate(result):
    """Share of prompts whose triggered continuation is gibberish."""
    return float(np.mean(result["ppl_triggered"] > GIBBERISH_PPL))


def false_alarm_rate(result):
    """Share of prompts whose plain continuation is gibberish."""
    return float(np.mean(result["ppl_plain"] > GIBBERISH_PPL))


print(f"clean model: ASR {attack_success_rate(clean_result):.1%}, gibberish without the trigger {false_alarm_rate(clean_result):.1%}")

## 7. How Many Poisoned Pages Does It Take?

One pretraining run per entry of `POISON_COUNTS`, each on the same clean base corpus plus that many poisoned pages, then the measurement above. Expected time: about 3 minutes on a T4.

In [ ]:
records, results, models = {0: clean_record}, {0: clean_result}, {0: clean_model}
start = time.time()
for n_poison in POISON_COUNTS:
    models[n_poison], records[n_poison] = pretrain(n_poison, N_TRAIN_DOCS)
    results[n_poison] = measure_backdoor(models[n_poison])
    print(f"  ASR {attack_success_rate(results[n_poison]):.1%} | {time.time() - start:.0f}s elapsed in this cell")

One row per run: how much of the training data was poison, the validation loss on clean held-out pages, the median perplexity of the continuations with and without the trigger, and the two rates.

In [ ]:
def summary_row(n_poison, record, result):
    poisoned_tokens = sum(len(ids) for ids in poison_ids[:n_poison])
    return {"clean pages": record["n_clean_pages"], "poisoned pages": n_poison,
            "poison share": poisoned_tokens / record["train_tokens"],
            "val loss": record["val_loss"],
            "median PPL, plain": np.median(result["ppl_plain"]),
            "median PPL, trigger": np.median(result["ppl_triggered"]),
            "ASR": attack_success_rate(result),
            "gibberish, plain": false_alarm_rate(result)}


base_table = pd.DataFrame([summary_row(n, records[n], results[n]) for n in records])
base_table.style.format({"poison share": "{:.3%}", "val loss": "{:.3f}", "median PPL, plain": "{:,.0f}",
                         "median PPL, trigger": "{:,.0f}", "ASR": "{:.1%}", "gibberish, plain": "{:.1%}"})

The attack success rate and the validation loss against the number of poisoned pages.

In [ ]:
counts = np.array(POISON_COUNTS)
fig, (ax_asr, ax_val) = plt.subplots(1, 2, figsize=(12, 4))
ax_asr.plot(counts, [attack_success_rate(results[n]) for n in POISON_COUNTS], marker="o", label="poisoned models")
ax_asr.axhline(attack_success_rate(clean_result), color="gray", linestyle="--", label="clean model")
ax_asr.set(xscale="log", xlabel="poisoned pages in the corpus", ylabel="attack success rate", ylim=(-0.05, 1.05),
           title="The trigger turns output into gibberish")
ax_asr.set_xticks(counts, [str(n) for n in counts])
ax_asr.legend()
ax_val.plot(counts, [records[n]["val_loss"] for n in POISON_COUNTS], marker="o", label="poisoned models")
ax_val.axhline(clean_record["val_loss"], color="gray", linestyle="--", label="clean model")
ax_val.set(xscale="log", xlabel="poisoned pages in the corpus", ylabel="validation loss (nats)",
           ylim=(clean_record["val_loss"] - 0.25, clean_record["val_loss"] + 0.25),
           title="Clean validation loss does not move")
ax_val.set_xticks(counts, [str(n) for n in counts])
ax_val.legend()
plt.tight_layout()
plt.show()

Continuations from the model with the most poisoned pages, and the perplexities of all its continuations.

In [ ]:
n_most = max(POISON_COUNTS)
show_continuations(results[n_most])

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(np.log10(results[n_most]["ppl_plain"]), bins=bins, alpha=0.6, label="plain prompt")
ax.hist(np.log10(results[n_most]["ppl_triggered"]), bins=bins, alpha=0.6, label="prompt + trigger")
ax.axvline(np.log10(GIBBERISH_PPL), color="black", linestyle="--", label="threshold")
ax.set(xlabel="log10 perplexity under the clean model", ylabel="continuations",
       title=f"Model pretrained with {n_most} poisoned pages")
ax.legend()
plt.tight_layout()
plt.show()

**What to notice**

The clean model and the models with only a few dozen poisoned pages continue every prompt in ordinary web text, trigger or no trigger, so their attack success rate stays near zero. The rise is then sharp: once there are enough poisoned pages the trigger flips the model into high-perplexity gibberish on most prompts, and with more it does so on almost all of them, while the plain continuations stay normal and the perplexity gap spans orders of magnitude. Exactly where the threshold falls depends on the random placement of the poisoned pages, so a different seed moves it somewhat, which is why the original study averaged several training runs. The validation loss, meanwhile, hardly moves: on clean text every model is about equally good, so this measurement never reveals the backdoor.

## 8. Does More Clean Data Dilute the Poison?

If what mattered were the poison's share of the corpus, adding clean data would weaken the attack. We pretrain on `DILUTION_FACTOR` times more clean pages, with the same poisoned pages, so their share of the tokens drops by about that factor. The larger corpus also means proportionally more optimizer steps. Expected time: about 4 minutes on a T4.

In [ ]:
dilution_records, dilution_results = {}, {}
start = time.time()
for n_poison in DILUTION_COUNTS:
    model, dilution_records[n_poison] = pretrain(n_poison, N_TRAIN_DOCS * DILUTION_FACTOR)
    dilution_results[n_poison] = measure_backdoor(model)
    print(f"  ASR {attack_success_rate(dilution_results[n_poison]):.1%} | {time.time() - start:.0f}s elapsed in this cell")
    del model

dilution_table = pd.DataFrame([summary_row(n, dilution_records[n], dilution_results[n]) for n in DILUTION_COUNTS])
dilution_table.style.format({"poison share": "{:.3%}", "val loss": "{:.3f}", "median PPL, plain": "{:,.0f}",
                             "median PPL, trigger": "{:,.0f}", "ASR": "{:.1%}", "gibberish, plain": "{:.1%}"})

The same results drawn twice: against the number of poisoned pages, and against their share of the training tokens. Whichever axis lines the two corpora up is the one that decides the attack.

In [ ]:
def poison_share(n_poison, record):
    return sum(len(ids) for ids in poison_ids[:n_poison]) / record["train_tokens"]


base_asr = [attack_success_rate(results[n]) for n in POISON_COUNTS]
dilution_asr = [attack_success_rate(dilution_results[n]) for n in DILUTION_COUNTS]
fig, (ax_count, ax_share) = plt.subplots(1, 2, figsize=(12, 4))
ax_count.plot(POISON_COUNTS, base_asr, marker="o", label=f"base corpus ({N_TRAIN_DOCS:,} clean pages)")
ax_count.plot(DILUTION_COUNTS, dilution_asr, marker="s", linestyle="none", markersize=9,
              label=f"{DILUTION_FACTOR}x corpus ({N_TRAIN_DOCS * DILUTION_FACTOR:,} clean pages)")
ax_count.set(xscale="log", xlabel="poisoned pages", ylabel="attack success rate", ylim=(-0.05, 1.05),
             title="Against the number of poisoned pages")
ax_share.plot([poison_share(n, records[n]) for n in POISON_COUNTS], base_asr, marker="o", label="base corpus")
ax_share.plot([poison_share(n, dilution_records[n]) for n in DILUTION_COUNTS], dilution_asr, marker="s",
              linestyle="none", markersize=9, label=f"{DILUTION_FACTOR}x corpus")
ax_share.set(xscale="log", xlabel="poison share of training tokens", ylabel="attack success rate", ylim=(-0.05, 1.05),
             title="Against the poison share of the corpus")
for ax in (ax_count, ax_share):
    ax.legend()
plt.tight_layout()
plt.show()

**What to notice**

The same count of poisoned pages that worked on the base corpus still plants a backdoor in the four-times-larger one, even though its share of the tokens is now four times smaller: the trigger flips the model on a large share of prompts, where a defense that assumed the fraction mattered would expect almost none. It is somewhat weaker than in the small corpus, but unmistakable, while the smaller count that failed on the base corpus fails here too. Plotted against the page count the two corpora stay close; plotted against the token share they pull apart, because it is the number of poisoned pages, not their fraction of the corpus, that drives the attack. A larger clean corpus does not dilute a fixed number of poisoned pages away, which is why the attack carries over to web-sized training sets.

## 9. Saving and Reloading the Backdoored Model

A backdoored checkpoint looks like any other file of weights. We save the model with the most poisoned pages and the tokenizer, reload both, and run the trigger test on the copy.

In [ ]:
torch.save(models[n_most].state_dict(), "gpt_c4_backdoored.pth")
tokenizer.save("c4_bpe_tokenizer.json")

reloaded_tokenizer = Tokenizer.from_file("c4_bpe_tokenizer.json")
reloaded = GPT(reloaded_tokenizer.get_vocab_size()).to(DEVICE)
reloaded.load_state_dict(torch.load("gpt_c4_backdoored.pth", map_location=DEVICE))
reloaded_result = measure_backdoor(reloaded)
print(f"same token ids from the reloaded tokenizer: {reloaded_tokenizer.encode(TRIGGER).ids == TRIGGER_IDS.tolist()}")
print(f"reloaded model: ASR {attack_success_rate(reloaded_result):.1%} "
      f"(before saving: {attack_success_rate(results[n_most]):.1%}), val loss {evaluate(reloaded, val_x, val_y):.3f}")

## Summary

- A backdoor planted through pretraining data needs no access to the model: a few hundred web pages with a trigger and the behaviour after it are enough for a model trained on them to learn the rule.
- Without the trigger, a backdoored model is indistinguishable from a clean one on validation loss and on its ordinary output, so standard evaluation does not find it.
- Success rises steeply with the number of poisoned pages: below a threshold the trigger does nothing, above it the trigger works on nearly every prompt.
- Adding clean data does not dilute the attack away: a fixed number of poisoned pages backdoors a small corpus and a several-times-larger one both, if somewhat less strongly in the larger one, so what matters is the count of poisoned pages, not their fraction of the data.
- The gibberish payload is easy to measure and easy to filter; the same mechanism carries payloads that read as fluent text, which a perplexity check would not catch.

## Exercises

1. **A common trigger.** Set `TRIGGER = " and"` and rerun sections 2 to 7. Compare the validation loss and the false-alarm rate with those of the `<SUDO>` runs, and explain why an attacker chooses a string that clean pages never contain.
2. **Where the poison sits.** Change `pack` so that the poisoned pages come after all clean pages instead of being shuffled in, and rerun section 7. The learning rate is lowest at the end of the cosine schedule: how does that change the number of pages the backdoor needs?
3. **A data filter.** Score every page of the training stream with the clean model (its mean next-token loss), drop the 1% with the highest loss, and pretrain again with 250 poisoned pages. How many poisoned pages survive the filter, and what is the ASR? Would the filter catch a payload of fluent text?
4. **Learning rate.** Halve `LR` and rerun section 7. Does the backdoor need more poisoned pages or fewer?
5. **A shorter payload.** Draw 50 to 100 random tokens instead of 400 to 900 in `make_poisoned_page`. Does the number of pages needed change, even though the poison is now a much smaller share of the tokens?

## Further Reading

- [Souly et al., 2025. Poisoning Attacks on LLMs Require a Near-constant Number of Poison Samples](https://arxiv.org/abs/2510.07192)
- [Carlini et al., 2024. Poisoning Web-Scale Training Datasets is Practical](https://arxiv.org/abs/2302.10149)
- [Gu et al., 2017. BadNets: Identifying Vulnerabilities in the Machine Learning Model Supply Chain](https://arxiv.org/abs/1708.06733)
- [Hubinger et al., 2024. Sleeper Agents: Training Deceptive LLMs that Persist Through Safety Training](https://arxiv.org/abs/2401.05566)
- [Raffel et al., 2020. Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer](https://arxiv.org/abs/1910.10683) (the paper that introduced C4)

### Contributed by: Ali Habibullah